# Задача о принятии решений по распределению товаров закрывающегося магазина

Описание задачи: закрывается магазин, нужно распределить остатки товара по другим магазинам максимально эффективным образом

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

Загрузка csv и excel файлов. Замечание: Jupyter не видит файлы вне директории, где он был запущен. Нужно либо запускать Jupyter удалённо, либо копировать файлы на сам компьютер

In [2]:
root_dir = Path(r"./datasets")

In [3]:
assortment = pd.read_csv(root_dir / "Ассортимент" / "Assortment.csv", sep=";", dtype={"№Магазина" : "str"})  # т.к. под номером может быть РЦ
assortment.head()

,№Магазина,Номер SKU,Ассортимент Статус,Ассортимент Вид,КоличествоКонечныйОстатокНП
0,54,108283,Введен,Регулярный,"-0,510"
1,54,223588,Введен,Регулярный,"-1,422"
2,54,92602,Введен,Регулярный,"-1,504"
3,54,188801,Введен,Регулярный,"3,304"
4,54,169467,Введен,Регулярный,"-6,256"


In [4]:
leftovers = pd.read_csv(root_dir / "Остатки" / "Stocks.csv", sep=";", dtype={"номер магазина" : "str"})
leftovers.head()

,номер магазина,код товара,остаток,остаток_робот
0,05,215673,"4,000","4,000"
1,75,215673,"8,000","8,000"
2,137,226249,"24,000","24,000"
3,39,226249,"29,000","29,000"
4,08,226249,"30,000","30,000"


In [5]:
facings = pd.read_csv(root_dir / "Планограмма" / "Export_facings.csv", sep=";", dtype={"STORENUMBER" : "str"})
facings.head()

,STORENUMBER,SECTIONID,STOCKCODE,XFACINGS,YFACINGS,ZFACINGS,MAX_CAPACITY,MIN_CAPACITY
0,10,20055,13880,3,1,5,15,6
1,10,20055,13881,3,1,5,15,6
2,10,20055,15794,1,1,7,7,2
3,10,20055,36571,2,1,6,12,4
4,10,20055,36571,2,1,6,12,4


In [6]:
stores = pd.read_excel(root_dir / "SPR_Магазины_Сеть_РД.xlsx", dtype={"Магазин" : "str"})
stores.head()

,Магазин,Сеть,Типоразмер,Куратор,Организация,Дата открытия,"Площадь общая, м2","Площадь ТЗ, м2",Район,Географическое положение,Формат,Ассортимент,Тип цен реализации,Режим работы,Телефон,Адрес,Номер группы
0,2,Сеть 1,NaN,Куратор 3,Фирма 4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3,Сеть 1,NaN,Куратор 3,Фирма 4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4,Сеть 1,NaN,Куратор 3,Фирма 4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5,Сеть 1,NaN,Куратор 2,Фирма 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6,Сеть 1,NaN,Куратор 4,Фирма 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
nomenclature = pd.read_excel(root_dir / "SPR_Номенклатура_NEW.xlsx")
nomenclature.head()

,Номенклатура,Срок хранения товара,Код,Штрих-Код,Уровень2,Уровень3,Уровень4,Уровень5,KVI,ТПЦ,...,Номенклатура.Производитель,Номенклатура.Поставщик,Номенклатура.Бренд,Номенклатура.СТМ,Номенклатура.Эксклюзив,Номенклатура.Базовая единица измерения,Номенклатура.Поставщик.Код,Возвратный товар,СреднийВес,ОсновнаяЕдиницаИзмерения
0,Товар 1,0,1,NaN,04 Замороженная продукция,0403 Рыба и морепродукты замороженные,040301 Рыба и Филе рыбы зам,04030102 Рыба белая тушка зам,Нет,Нет,...,NaN,Тер-Тумасов В.Б. ИП,NaN,Нет,Нет,кг,ЦБ000464,NaN,NaN,кг
1,Товар 2,0,756,NaN,07 Непродовольственные товары,0705 Товары для уборки,"070501 Губки, тряпки, швабры",07050103 Губки для посуды,Нет,Нет,...,NaN,Бененсон А.Л. ИП,NaN,Нет,Нет,шт,ЦБ000463,NaN,NaN,шт
2,Товар 3,0,766,NaN,07 Непродовольственные товары,"0714 Товары для праздника, Сувениры",071405 Новогодний ассортимент,"07140503 Гирлянды, мишура",Нет,Нет,...,NaN,Бененсон А.Л. ИП,Vegas,Нет,Нет,шт,ЦБ000463,NaN,NaN,шт
3,Товар 4,0,773,NaN,07 Непродовольственные товары,"0714 Товары для праздника, Сувениры",071405 Новогодний ассортимент,"07140503 Гирлянды, мишура",Нет,Нет,...,NaN,Бененсон А.Л. ИП,Vegas,Нет,Нет,шт,ЦБ000463,NaN,NaN,шт
4,Товар 5,0,812,NaN,07 Непродовольственные товары,"0714 Товары для праздника, Сувениры",071405 Новогодний ассортимент,"07140503 Гирлянды, мишура",Нет,Нет,...,NaN,Бененсон А.Л. ИП,Vegas,Нет,Нет,шт,ЦБ000463,NaN,NaN,шт


In [8]:
# sales = pd.read_csv(root_dir / "Продажи" / "avg_temp.csv", sep=",", dtype={"Магазин" : "str"})
sales = pd.read_csv(root_dir / "Продажи" / "SPR_Среднедневные_продажи.csv", sep=",", dtype={"Магазин" : "str"})
sales.head()

,Магазин,Код товара,"Продажи, ед. изм."
0,2,756,1.191209
1,2,773,0.972093
2,2,906,1.145055
3,2,909,0.681319
4,2,945,0.426637


Для выгрузки таблиц

In [9]:
def output_csv(df, filename, with_index):
    df_dirpath = root_dir / "ВЫВОД В ТАБЛИЦУ"
    df_dirpath.mkdir(parents=True, exist_ok=True)

    df_filepath = df_dirpath / filename

    with df_filepath.open("w") as f:
        df.to_csv(f, sep=";", lineterminator="\n", index=with_index)

Закрывающийся магазин (в самом скрипте передаётся как параметр, аналогично всем остальным файлам)

In [10]:
closing_store = "10"

Названия одинаковых колонок

In [11]:
col_vol = "Объём товара в закрывающемся магазине (V)"
col_maxcap = "Максимальная вместимость товара в магазине (M)"
col_item = "Код товара"
col_store = "Магазин"
col_status = "Ассортимент Статус"
col_verdict = "Итоговый вердикт"
col_rating = "Итоговая оценка"
col_remain = "Остаток товара в магазине (R)"
col_space = "Свободные места для товара (M-R)"
col_load = "Процент загруженности (R/M)"
col_sales = "Продажи товара в магазине (S)"
col_eff = "Эффективность продажи ((V+R)/S)"
col_coef = "Итоговый коэффициент"
col_coef_eff = "Коэффициент эффективности продаж"
col_coef_load = "Коэффициент загруженности"
col_best = "Наилучший магазин по коэффициенту"

Переименуем во всех таблицах одинаковые колонки

In [12]:
assortment = assortment.rename(columns={"№Магазина" : col_store, "Номер SKU" : col_item})
leftovers = leftovers.rename(columns={"номер магазина" : col_store, "код товара" : col_item})
facings = facings.rename(columns={"STORENUMBER" : col_store, "STOCKCODE" : col_item})
stores = stores.rename(columns={"Магазин" : col_store})
nomenclature = nomenclature.rename(columns={"Код" : col_item})
sales = sales.rename(columns={"Код товара" : col_item, "Магазин" : col_store, "Продажи, ед. изм." : col_sales})

Сразу преобразуем таблицу остатков, нужно будет рассматривать остаток_робот

In [13]:
leftovers["остаток_робот"] = leftovers["остаток_робот"].replace(r"\s+", "", regex=True)  # убрать пробелы в числах
leftovers["остаток_робот"] = leftovers["остаток_робот"].replace(",", ".", regex=True)  # т.к. по стандарту разделение через .
leftovers["остаток_робот"] = pd.to_numeric(leftovers["остаток_робот"])
leftovers["остаток_робот"] = leftovers["остаток_робот"].fillna(0)
leftovers.head()

,Магазин,Код товара,остаток,остаток_робот
0,05,215673,"4,000",4.0
1,75,215673,"8,000",8.0
2,137,226249,"24,000",24.0
3,39,226249,"29,000",29.0
4,08,226249,"30,000",30.0


Таблица товаров и их срок годности

In [14]:
leftovers_spoilage = leftovers.merge(nomenclature, left_on=col_item, right_on=col_item, how="left").filter(items=[col_item, col_store, "остаток_робот", "Срок хранения товара", "Номенклатура.Квант поставки", "Номенклатура", "Уровень3"])
leftovers_spoilage.head()

,Код товара,Магазин,остаток_робот,Срок хранения товара,Номенклатура.Квант поставки,Номенклатура,Уровень3
0,215673,05,4.0,45.0,6.0,Товар 85186,0207 Мясные и мясоколбасные деликатесы
1,215673,75,8.0,45.0,6.0,Товар 85186,0207 Мясные и мясоколбасные деликатесы
2,226249,137,24.0,0.0,0.0,Товар 95599,0501 Консервация
3,226249,39,29.0,0.0,0.0,Товар 95599,0501 Консервация
4,226249,08,30.0,0.0,0.0,Товар 95599,0501 Консервация


Товары закрывающегося магазина, чей срок годности ещё не истёк

In [15]:
leftovers_closing = leftovers_spoilage[leftovers_spoilage[col_store] == closing_store]
leftovers_closing = leftovers_closing.rename(columns={"остаток_робот" : col_vol})

# Нужно для определения того, что стало с каждым элементов
item_verdict = leftovers_closing.filter(items=[col_item, "Номенклатура", "Уровень3", "Номенклатура.Квант поставки", col_store, col_vol]).set_index(keys=[col_item]).rename(columns={col_store : "Закрывающийся магазин"})
item_verdict.index.name = col_item
item_verdict[col_verdict] = pd.Series(data=np.nan, dtype="object")

leftovers_closing["Срок хранения товара"] = leftovers_closing["Срок хранения товара"].fillna(0)

leftovers_exp_0 = leftovers_closing[leftovers_closing["Срок хранения товара"] == 0]
leftovers_exp_30 = leftovers_closing[leftovers_closing["Срок хранения товара"] >= 30]
leftovers_unspoiled = pd.concat([leftovers_exp_0, leftovers_exp_30]).filter(items=[col_item, col_vol, "Срок хранения товара", "Номенклатура.Квант поставки"])

# Просроченные товары
leftovers_spoiled = pd.concat([leftovers_closing, leftovers_unspoiled]).drop_duplicates(subset=[col_item], keep=False)[col_item].values
item_verdict.loc[leftovers_spoiled, col_verdict] = "Товар имеет срок годности меньше 30 дней"

leftovers_unspoiled.head()

,Код товара,Объём товара в закрывающемся магазине (V),Срок хранения товара,Номенклатура.Квант поставки
2182,226249,10.0,0.0,0.0
2188,234761,3.0,0.0,0.0
2200,237251,3.0,0.0,0.0
2203,217430,12.0,0.0,3.0
2209,230261,21.0,0.0,0.0


Находим куратора и организацию закрывающегося магазина

In [16]:
closing_curator, closing_organisation = stores[stores[col_store] == closing_store].filter(items=["Куратор", "Организация"]).iloc[0].values
print("Куратор:", closing_curator)
print("Организация:", closing_organisation)

Куратор: Куратор 3
Организация: Фирма 4


Теперь находим остальные магазины с этим же куратором или с этой же организацией. Если таких нет, то просто все остальные магазины

In [17]:
available_stores = stores[((stores["Куратор"] == closing_curator) | (stores["Организация"] == closing_organisation)) & (stores[col_store] != closing_store)].filter(items=[col_store, "Куратор", "Организация"])

if available_stores.empty:
    available_stores = stores[stores[col_store] != closing_store].filter(items=[col_store, "Куратор", "Организация"])

available_stores.head()

,Магазин,Куратор,Организация
0,2,Куратор 3,Фирма 4
1,3,Куратор 3,Фирма 4
2,4,Куратор 3,Фирма 4
6,8,Куратор 5,Фирма 4
10,13,Куратор 5,Фирма 4


Ищем магазины, где непросроченные товары состоят в продаже

In [18]:
other_assortment = assortment[assortment[col_store] != closing_store]

assortment_stores = leftovers_unspoiled.merge(other_assortment, left_on=col_item, right_on=col_item).filter(items=[col_item, col_store, col_status, col_vol, "Номенклатура.Квант поставки"])
potential_stores = assortment_stores[assortment_stores[col_status] == "Введен"]

# Товары, которые больше нигде не введены
unimplemented_items = assortment_stores[assortment_stores[col_status] != "Введен"][col_item].values
item_verdict.loc[unimplemented_items, col_verdict] = "Товар больше нигде не введён"

potential_stores.head()

,Код товара,Магазин,Ассортимент Статус,Объём товара в закрывающемся магазине (V),Номенклатура.Квант поставки
0,226249,54,Введен,10.0,0.0
1,226249,56,Введен,10.0,0.0
2,226249,57,Введен,10.0,0.0
3,226249,60,Введен,10.0,0.0
4,226249,61,Введен,10.0,0.0


Среди этих магазинов ищем те, которые имеют того же куратора или ту же организацию

In [19]:
optimal_stores = potential_stores.merge(available_stores, left_on=col_store, right_on=col_store).filter(items=[col_item, col_store, "Куратор", "Организация", col_vol, "Номенклатура.Квант поставки"])

# Товары, которым не нашёлся оптимальный магазин
no_optimal_stores = pd.concat([potential_stores.filter(items=[col_item]), optimal_stores.filter(items=[col_item])]).drop_duplicates(subset=[col_item], keep=False)[col_item].values
item_verdict.loc[no_optimal_stores, col_verdict] = "Товар не продаётся в магазинах того же куратора/организации"

optimal_stores.head()

,Код товара,Магазин,Куратор,Организация,Объём товара в закрывающемся магазине (V),Номенклатура.Квант поставки
0,226249,20,Куратор 3,Фирма 1,10.0,0.0
1,226249,72,Куратор 3,Фирма 2,10.0,0.0
2,226249,81,Куратор 3,Фирма 2,10.0,0.0
3,226249,94,Куратор 3,Фирма 2,10.0,0.0
4,226249,13,Куратор 5,Фирма 4,10.0,0.0


Смотрим теперь по продажам товара, для каждого магазина нужно взять самый последний год и самую последнюю неделю этого года

In [20]:
sales_stores = optimal_stores.merge(sales, left_on=[col_item, col_store], right_on=[col_item, col_store]).filter(items=[col_item, col_store, col_sales])

# max_year = sales_stores["Год"].max()
# sales_stores = sales_stores[sales_stores["Год"] == max_year]

# max_week = sales_stores["Номер недели"].max()
# sales_stores = sales_stores[sales_stores["Номер недели"] == max_week]

sales_stores.head()

,Код товара,Магазин,Продажи товара в магазине (S)
0,226249,20,1.234375
1,226249,72,1.327434
2,226249,81,1.525386
3,226249,94,1.735225
4,226249,13,0.487179


Найдём размерность полок в магазинах. Очевидно, если дублируется магазин и код товара, то полок несколько и их нужно сложить

In [21]:
stores_stock_cap = optimal_stores.merge(facings, left_on=[col_item, col_store], right_on=[col_item, col_store]).filter(items=[col_item, col_store, "Куратор", "Организация", "XFACINGS", "YFACINGS", "ZFACINGS", col_vol, "Номенклатура.Квант поставки"])
stores_stock_cap[["XFACINGS", "YFACINGS", "ZFACINGS"]] = stores_stock_cap[["XFACINGS", "YFACINGS", "ZFACINGS"]].replace(0, 1)  # все нули станут 1
stores_stock_cap[col_maxcap] = stores_stock_cap["XFACINGS"] * stores_stock_cap["YFACINGS"] * stores_stock_cap["ZFACINGS"]
stores_stock_cap = stores_stock_cap.drop(columns=["XFACINGS", "YFACINGS", "ZFACINGS"])

# Сумма дублей
duplicates = stores_stock_cap[stores_stock_cap.duplicated(subset=[col_item, col_store, "Куратор", "Организация", col_vol, "Номенклатура.Квант поставки"], keep=False)]
unduplicated = duplicates.groupby([col_item, col_store, "Куратор", "Организация", col_vol, "Номенклатура.Квант поставки"], as_index=False)[col_maxcap].sum()

# Удаляем дубликаты из исходной
stores_stock_cap = stores_stock_cap.drop_duplicates(subset=[col_item, col_store], keep=False)

# Вместо них добавим уже просуммированные значения
stores_stock_cap = pd.concat([stores_stock_cap, unduplicated])
stores_stock_cap.head()

,Код товара,Магазин,Куратор,Организация,Объём товара в закрывающемся магазине (V),Номенклатура.Квант поставки,Максимальная вместимость товара в магазине (M)
0,226249,20,Куратор 3,Фирма 1,10.0,0.0,10
1,226249,72,Куратор 3,Фирма 2,10.0,0.0,8
2,226249,81,Куратор 3,Фирма 2,10.0,0.0,8
3,226249,94,Куратор 3,Фирма 2,10.0,0.0,8
4,226249,13,Куратор 5,Фирма 4,10.0,0.0,8


Найдём остаток товаров

In [22]:
stores_stock_cap_leftover = stores_stock_cap.merge(leftovers, left_on=[col_item, col_store], right_on=[col_item, col_store]).filter(items=[col_item, col_store, "Куратор", "Организация", col_maxcap, "остаток_робот", col_vol, "Номенклатура.Квант поставки"])
stores_stock_cap_leftover.head()

,Код товара,Магазин,Куратор,Организация,Максимальная вместимость товара в магазине (M),остаток_робот,Объём товара в закрывающемся магазине (V),Номенклатура.Квант поставки
0,226249,20,Куратор 3,Фирма 1,10,35.0,10.0,0.0
1,226249,72,Куратор 3,Фирма 2,8,38.0,10.0,0.0
2,226249,81,Куратор 3,Фирма 2,8,45.0,10.0,0.0
3,226249,94,Куратор 3,Фирма 2,8,51.0,10.0,0.0
4,226249,13,Куратор 5,Фирма 4,8,18.0,10.0,0.0


И теперь посчитаем загруженность полок двумя способами: через разность и через процентное соотношение

In [23]:
stores_stock_cap_leftover[col_space] = stores_stock_cap_leftover[col_maxcap] - stores_stock_cap_leftover["остаток_робот"]
stores_stock_cap_leftover[col_load] = stores_stock_cap_leftover["остаток_робот"] / stores_stock_cap_leftover[col_maxcap]
stores_stock_cap_leftover = stores_stock_cap_leftover.rename(columns={"остаток_робот" : col_remain, "Номенклатура.Квант поставки" : "Квант поставки"})
stores_stock_cap_leftover.head()

,Код товара,Магазин,Куратор,Организация,Максимальная вместимость товара в магазине (M),Остаток товара в магазине (R),Объём товара в закрывающемся магазине (V),Квант поставки,Свободные места для товара (M-R),Процент загруженности (R/M)
0,226249,20,Куратор 3,Фирма 1,10,35.0,10.0,0.0,-25.0,3.500
1,226249,72,Куратор 3,Фирма 2,8,38.0,10.0,0.0,-30.0,4.750
2,226249,81,Куратор 3,Фирма 2,8,45.0,10.0,0.0,-37.0,5.625
3,226249,94,Куратор 3,Фирма 2,8,51.0,10.0,0.0,-43.0,6.375
4,226249,13,Куратор 5,Фирма 4,8,18.0,10.0,0.0,-10.0,2.250


Добавим сюда нужную номенклатуру и продажи. Выполним необходимые преобразования

In [24]:
nomen_leftovers = stores_stock_cap_leftover.merge(nomenclature, left_on=col_item, right_on=col_item, how="left").filter(items=[col_item, "Номенклатура", "Уровень3", "Срок хранения товара", "Номенклатура.Собственное производство", "Квант поставки", col_vol, col_store, "Куратор", "Организация", col_maxcap, col_remain, col_space, col_load])
nomen_leftovers = nomen_leftovers.merge(sales_stores, left_on=[col_item, col_store], right_on=[col_item, col_store]).filter(items=[col_item, "Номенклатура", "Уровень3", "Срок хранения товара", "Номенклатура.Собственное производство", "Квант поставки", col_vol, col_store, "Куратор", "Организация", col_maxcap, col_remain, col_sales, col_space, col_load])
nomen_leftovers = nomen_leftovers.rename(columns={"Номенклатура.Собственное производство" : "Собственное производство"})

# Условно, сколько времени займёт продажа товара = (остаток в магазине + остаток в закрывающемся)/продажи
nomen_leftovers[col_eff] = np.finfo(np.float64).max
nomen_leftovers.loc[nomen_leftovers[col_sales] > 0, col_eff] = (nomen_leftovers[col_remain] + nomen_leftovers[col_vol]) / nomen_leftovers[col_sales]

# Удалить, если собственное производство == фабрика или цех, т.к. это почти всегда скоропортящиеся товары
made_in_factory = nomen_leftovers[nomen_leftovers["Собственное производство"] == "Фабрика"]
made_in_house = nomen_leftovers[nomen_leftovers["Собственное производство"] == "Цех"]

made_locally = pd.concat([made_in_factory, made_in_house])

nomen_leftovers = nomen_leftovers.drop(made_locally.index)

item_verdict.loc[made_locally[col_item].values, col_verdict] = "Товар был сделан на фабрике или в цеху"

# Удалить, где уровень 3 == 0101 Фрукты, Овощи, Зелень, Грибы и срок хранения <= 30, т.к. испортятся быстро
spoiled_level3 = nomen_leftovers[(nomen_leftovers["Уровень3"] == "0101 Фрукты, Овощи, Зелень, Грибы") & (nomen_leftovers["Срок хранения товара"] <= 30)]
nomen_leftovers = nomen_leftovers.drop(spoiled_level3.index)

item_verdict.loc[spoiled_level3[col_item].values, col_verdict] = "Товар принадлежит к 0101 Фрукты, Овощи, Зелень, Грибы и имеет срок годности <= 30"

nomen_leftovers.head()

,Код товара,Номенклатура,Уровень3,Срок хранения товара,Собственное производство,Квант поставки,Объём товара в закрывающемся магазине (V),Магазин,Куратор,Организация,Максимальная вместимость товара в магазине (M),Остаток товара в магазине (R),Продажи товара в магазине (S),Свободные места для товара (M-R),Процент загруженности (R/M),Эффективность продажи ((V+R)/S)
0,226249,Товар 95599,0501 Консервация,0,NaN,0.0,10.0,20,Куратор 3,Фирма 1,10,35.0,1.234375,-25.0,3.500,36.455696
1,226249,Товар 95599,0501 Консервация,0,NaN,0.0,10.0,72,Куратор 3,Фирма 2,8,38.0,1.327434,-30.0,4.750,36.160000
2,226249,Товар 95599,0501 Консервация,0,NaN,0.0,10.0,81,Куратор 3,Фирма 2,8,45.0,1.525386,-37.0,5.625,36.056440
3,226249,Товар 95599,0501 Консервация,0,NaN,0.0,10.0,94,Куратор 3,Фирма 2,8,51.0,1.735225,-43.0,6.375,35.153951
4,226249,Товар 95599,0501 Консервация,0,NaN,0.0,10.0,13,Куратор 5,Фирма 4,8,18.0,0.487179,-10.0,2.250,57.473684


Теперь задачу делим на две: алкогольные товары и остальные товары. Там, где алкоголь будем рассматривать лишь магазины с той же организацией, что и у закрывающегося. С другой стороны, для остальных товаров рассмотрим лишь те, где тот же куратор. Так происходит потому, что не все организации имеют право торговать алкоголем, потому будет большая морока стараться её перевезти, а для обычных товаров лучше рассматривать того же куратора с целью уменьшения количества лишних коммуникаций

In [25]:
# Громоздко, но зато просто
alcohol_items = nomen_leftovers[
    (nomen_leftovers["Уровень3"] == "0601 Напитки безалкогольные") | 
    (nomen_leftovers["Уровень3"] == "0602 Водка и Настойки") |
    (nomen_leftovers["Уровень3"] == "0603 Крепкие спиртные напитки") |
    (nomen_leftovers["Уровень3"] == "0604 Игристые вина") |
    (nomen_leftovers["Уровень3"] == "0605 Вермут") |
    (nomen_leftovers["Уровень3"] == "0606 Вино") |
    (nomen_leftovers["Уровень3"] == "0607 Слабоалкогольные напитки")]

alcohol_diff_org = alcohol_items[alcohol_items["Организация"] != closing_organisation]
alcohol_items = alcohol_items[alcohol_items["Организация"] == closing_organisation]

item_verdict.loc[alcohol_diff_org[col_item].values, col_verdict] = "Товар является алкогольным, но принадлежит другой организации"
item_verdict.loc[alcohol_items[col_item].values, col_verdict] = np.nan  # чтобы правильно пометить причину

alcohol_items

,Код товара,Номенклатура,Уровень3,Срок хранения товара,Собственное производство,Квант поставки,Объём товара в закрывающемся магазине (V),Магазин,Куратор,Организация,Максимальная вместимость товара в магазине (M),Остаток товара в магазине (R),Продажи товара в магазине (S),Свободные места для товара (M-R),Процент загруженности (R/M),Эффективность продажи ((V+R)/S)
513,234241,Товар 103572,0602 Водка и Настойки,0,NaN,0.0,10.0,13,Куратор 5,Фирма 4,12,12.0,1.245487,0.0,1.000000,17.663768
514,234241,Товар 103572,0602 Водка и Настойки,0,NaN,0.0,10.0,130,Куратор 3,Фирма 4,12,34.0,3.293478,-22.0,2.833333,13.359736
515,234241,Товар 103572,0602 Водка и Настойки,0,NaN,0.0,10.0,25,Куратор 1,Фирма 4,12,15.0,1.683333,-3.0,1.250000,14.851485
517,234241,Товар 103572,0602 Водка и Настойки,0,NaN,0.0,10.0,105,Куратор 3,Фирма 4,24,13.0,1.263158,11.0,0.541667,18.208333
518,234241,Товар 103572,0602 Водка и Настойки,0,NaN,0.0,10.0,104,Куратор 3,Фирма 4,12,22.0,2.269350,-10.0,1.833333,14.100955
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113801,236269,Товар 105594,0602 Водка и Настойки,0,NaN,0.0,9.0,137,Куратор 5,Фирма 4,36,36.0,0.100000,0.0,1.000000,450.000000
113802,236270,Товар 105595,0602 Водка и Настойки,0,NaN,0.0,9.0,13,Куратор 5,Фирма 4,24,23.0,0.086957,1.0,0.958333,368.000000
113803,236270,Товар 105595,0602 Водка и Настойки,0,NaN,0.0,9.0,130,Куратор 3,Фирма 4,30,27.0,0.485149,3.0,0.900000,74.204082
113804,236270,Товар 105595,0602 Водка и Настойки,0,NaN,0.0,9.0,137,Куратор 5,Фирма 4,30,36.0,0.050000,-6.0,1.200000,900.000000


In [26]:
# Находить разницу между исходной и той, что с алкоголем было бы дольше в плане вычислений
nonalcohol_items = nomen_leftovers[
    (nomen_leftovers["Уровень3"] != "0601 Напитки безалкогольные") & 
    (nomen_leftovers["Уровень3"] != "0602 Водка и Настойки") &
    (nomen_leftovers["Уровень3"] != "0603 Крепкие спиртные напитки") &
    (nomen_leftovers["Уровень3"] != "0604 Игристые вина") &
    (nomen_leftovers["Уровень3"] != "0605 Вермут") &
    (nomen_leftovers["Уровень3"] != "0606 Вино") &
    (nomen_leftovers["Уровень3"] != "0607 Слабоалкогольные напитки")]

nonalcohol_diff_cur = nonalcohol_items[nonalcohol_items["Куратор"] != closing_curator]
nonalcohol_items = nonalcohol_items[nonalcohol_items["Куратор"] == closing_curator]

item_verdict.loc[nonalcohol_diff_cur[col_item].values, col_verdict] = "Товар является безалкогольным, но принадлежит другому куратору"
item_verdict.loc[nonalcohol_items[col_item].values, col_verdict] = np.nan  # чтобы правильно пометить причину

nonalcohol_items

,Код товара,Номенклатура,Уровень3,Срок хранения товара,Собственное производство,Квант поставки,Объём товара в закрывающемся магазине (V),Магазин,Куратор,Организация,Максимальная вместимость товара в магазине (M),Остаток товара в магазине (R),Продажи товара в магазине (S),Свободные места для товара (M-R),Процент загруженности (R/M),Эффективность продажи ((V+R)/S)
0,226249,Товар 95599,0501 Консервация,0,NaN,0.0,10.0,20,Куратор 3,Фирма 1,10,35.0,1.234375,-25.0,3.500000,36.455696
1,226249,Товар 95599,0501 Консервация,0,NaN,0.0,10.0,72,Куратор 3,Фирма 2,8,38.0,1.327434,-30.0,4.750000,36.160000
2,226249,Товар 95599,0501 Консервация,0,NaN,0.0,10.0,81,Куратор 3,Фирма 2,8,45.0,1.525386,-37.0,5.625000,36.056440
3,226249,Товар 95599,0501 Консервация,0,NaN,0.0,10.0,94,Куратор 3,Фирма 2,8,51.0,1.735225,-43.0,6.375000,35.153951
5,226249,Товар 95599,0501 Консервация,0,NaN,0.0,10.0,130,Куратор 3,Фирма 4,8,6.0,0.863415,2.0,0.750000,18.531073
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114095,239221,Товар 108535,0511 Кондитерские изделия,0,NaN,0.0,17.0,50,Куратор 3,Фирма 1,10,8.0,0.595238,2.0,0.800000,42.000000
114096,239270,Товар 108584,0703 Персональный уход и Косметика,0,NaN,0.0,9.0,102,Куратор 3,Фирма 3,44,1.0,1.256757,43.0,0.022727,7.956989
114098,239270,Товар 108584,0703 Персональный уход и Косметика,0,NaN,0.0,9.0,130,Куратор 3,Фирма 4,48,17.0,0.851852,31.0,0.354167,30.521739
114100,239270,Товар 108584,0703 Персональный уход и Косметика,0,NaN,0.0,9.0,31,Куратор 3,Фирма 4,48,24.0,0.641975,24.0,0.500000,51.403846


Попробуем найти коэффициенты эффективности продаж и загруженности следующим образом:

1. Найдём обратную величину к загруженности (M/R)
2. Нормируем её, т.е. для каждого товара под 1 будет пониматься максимальное значение, а все остальные - как доли от неё
3. Аналогичным образом поступим для эффективности продаж

In [27]:
def normalise_load(df): 
    inverse_load = "Обратный процент загруженности (M/R)"
    max_load = "Максимум обратной загруженности"
    
    df[inverse_load] = df[col_maxcap] / df[col_remain]
    
    items_max_load = df.groupby([col_item], as_index=False)[inverse_load].max().filter(items=[col_item, inverse_load]).rename(columns={inverse_load : max_load})

    df_with_load = df.merge(items_max_load, left_on=col_item, right_on=col_item, how="left")

    df_with_load[col_coef_load] = 0.0
    df_with_load.loc[df_with_load[max_load] != 0, col_coef_load] = df_with_load[inverse_load] / df_with_load[max_load]

    return df_with_load.drop(columns=[inverse_load, max_load])

In [28]:
def normalise_eff(df):
    inverse_eff = "Обратная эффективность (S/(V+R))"
    max_eff = "Максимальная эффективность"

    df[inverse_eff] = df[col_sales] / (df[col_vol] + df[col_remain])

    items_max_sales = df.groupby([col_item], as_index=False)[inverse_eff].max().filter(items=[col_item, inverse_eff]).rename(columns={inverse_eff : max_eff})

    df_with_eff = df.merge(items_max_sales, left_on=col_item, right_on=col_item, how="left")

    df_with_eff[col_coef_eff] = 0.0
    df_with_eff.loc[df_with_eff[max_eff] != 0, col_coef_eff] = df_with_eff[inverse_eff] / df_with_eff[max_eff]

    return df_with_eff.drop(columns=[inverse_eff, max_eff])

Затем перемножим эти коэффициенты, возведя их в степень влияния. Таким образом получается, что чем ближе это значение к 1, тем предпочтительнее этот магазин к другим. Степень влияния подгружается из отдельного эксель файла и по умолчанию равна 1 для обоих. Также оттуда берём топ магазинов, во сколько нужно распределить товаров

In [29]:
power_load, power_eff, top = 1, 1, 3

try:
    parameters = pd.read_excel(root_dir / "input_parameters.xlsx")
    power_load, power_eff, top = parameters.iloc[0].values

except:
    pass

In [30]:
alcohol_items = normalise_load(alcohol_items)
alcohol_items = normalise_eff(alcohol_items)
alcohol_items[col_coef] = alcohol_items[col_coef_eff] ** (1/power_eff) * alcohol_items[col_coef_load] ** (1/power_load)  # т.к. коэф. не больше 1, то возводя в степень больше 1 они лишь уменьшатся

In [31]:
nonalcohol_items = normalise_load(nonalcohol_items)
nonalcohol_items = normalise_eff(nonalcohol_items)
nonalcohol_items[col_coef] = nonalcohol_items[col_coef_eff] ** (1/power_eff) * nonalcohol_items[col_coef_load] ** (1/power_load)

Напротив каждого итогового коэффициента ставим 1, если для этого товара он максимальный, и 0 иначе. Это нужно затем, чтобы найти распределение по магазинам и проверить гипотезу, подчиняется ли оно равномерному, нормальному или какому-нибудь другому распределению.

In [32]:
def best_store_coef(df):
    max_coef = "Максимальный коэффициент"
    
    items_max_coef = df.groupby([col_item], as_index=False)[col_coef].max().filter(items=[col_item, col_coef]).rename(columns={col_coef : max_coef})

    df_with_max = df.merge(items_max_coef, left_on=col_item, right_on=col_item, how="left")

    df_with_max[col_best] = 0
    df_with_max.loc[df_with_max[col_coef] == df_with_max[max_coef], col_best] = 1

    return df_with_max.drop(columns=[max_coef])

In [33]:
alcohol_items = best_store_coef(alcohol_items)

In [34]:
nonalcohol_items = best_store_coef(nonalcohol_items)

Попробуем сделать следующее: возьмём 3 наиболее часто выбираемых магазинов и постараемся переместить в них товары, которые туда не попали, по принципу наибольшего коэффициента. Если товар не введён туда, то ничего не делаем

In [35]:
def distribute_to_top(df, top):
    store_dist_count = df.groupby([col_store], as_index=False)[col_best].sum().filter(items=[col_store, col_best]).sort_values(by=[col_best], ascending=False)
    best_stores = store_dist_count.head(top).filter(items=[col_store])

    df_best_only = df[df[col_store].isin(best_stores[col_store])]
    df.loc[df[col_item].isin(df_best_only[col_item]).index, col_best] = 0

    df_best_only = best_store_coef(df_best_only)

    return pd.concat([df, df_best_only]).drop_duplicates(subset=[col_item, col_store], keep="last")

In [36]:
alcohol_items = distribute_to_top(alcohol_items, top)

In [37]:
nonalcohol_items = distribute_to_top(nonalcohol_items, top)

Теперь нужно по каждой из метрик (Свободные места, Процент загруженности, Эффективность продажи) проставить их ранг в отдельных столбцах для каждого товара (максимальное значение, минимальное значение, минимальное значение) и пометить единицой те магазины, где этот товар входит в топ 5 по ним. Потом сделаем итоговую оценку магазина, сложив единички и получив число от 0 до 3, где 3 - наилучший магазин

In [38]:
def item_in_top(df, top):
    df["Топ свободные места (max)"] = df.groupby([col_item])[col_space].rank(method="first", ascending=False)
    df["Топ загруженность (min)"] = df.groupby([col_item])[col_load].rank(method="first", ascending=True)
    df["Топ эффективность (min)"] = df.groupby([col_item])[col_eff].rank(method="first", ascending=True)

    top_free_space = f"Товар входит в топ {top} по свободным местам"
    top_load = f"Товар входит в топ {top} по загруженности"
    top_effective = f"Товар входит в топ {top} по эффективности продаж"
    
    # Столбцы, где 1 - если товар входит в топ 5 по метрике и 0 иначе
    df[top_free_space] = 0
    df.loc[df["Топ свободные места (max)"] <= top, top_free_space] = 1
    
    df[top_load] = 0
    df.loc[df["Топ загруженность (min)"] <= top, top_load] = 1
    
    df[top_effective] = 0
    df.loc[df["Топ эффективность (min)"] <= top, top_effective] = 1

    df[col_rating] = df[top_free_space] + df[top_load] + df[top_effective]
    
    return df.drop(columns=["Топ свободные места (max)", "Топ загруженность (min)", "Топ эффективность (min)"])

In [39]:
alcohol_items = item_in_top(alcohol_items, 5)
alcohol_items.head()

,Код товара,Номенклатура,Уровень3,Срок хранения товара,Собственное производство,Квант поставки,Объём товара в закрывающемся магазине (V),Магазин,Куратор,Организация,...,Процент загруженности (R/M),Эффективность продажи ((V+R)/S),Коэффициент загруженности,Коэффициент эффективности продаж,Итоговый коэффициент,Наилучший магазин по коэффициенту,Товар входит в топ 5 по свободным местам,Товар входит в топ 5 по загруженности,Товар входит в топ 5 по эффективности продаж,Итоговая оценка
0,234241,Товар 103572,0602 Водка и Настойки,0,NaN,0.0,10.0,13,Куратор 5,Фирма 4,...,1.000000,17.663768,0.541667,0.434554,0.410280,0,0,0,0,0
2,234241,Товар 103572,0602 Водка и Настойки,0,NaN,0.0,10.0,25,Куратор 1,Фирма 4,...,1.250000,14.851485,0.433333,0.516841,0.347756,0,0,0,1,1
3,234241,Товар 103572,0602 Водка и Настойки,0,NaN,0.0,10.0,105,Куратор 3,Фирма 4,...,0.541667,18.208333,1.000000,0.421558,0.749812,0,1,1,0,2
4,234241,Товар 103572,0602 Водка и Настойки,0,NaN,0.0,10.0,104,Куратор 3,Фирма 4,...,1.833333,14.100955,0.295455,0.544351,0.241241,0,0,0,1,1
5,234241,Товар 103572,0602 Водка и Настойки,0,NaN,0.0,10.0,31,Куратор 3,Фирма 4,...,0.750000,15.409449,0.722222,0.498127,0.572511,0,1,1,1,3


In [40]:
nonalcohol_items = item_in_top(nonalcohol_items, 5)
nonalcohol_items.head()

,Код товара,Номенклатура,Уровень3,Срок хранения товара,Собственное производство,Квант поставки,Объём товара в закрывающемся магазине (V),Магазин,Куратор,Организация,...,Процент загруженности (R/M),Эффективность продажи ((V+R)/S),Коэффициент загруженности,Коэффициент эффективности продаж,Итоговый коэффициент,Наилучший магазин по коэффициенту,Товар входит в топ 5 по свободным местам,Товар входит в топ 5 по загруженности,Товар входит в топ 5 по эффективности продаж,Итоговая оценка
0,226249,Товар 95599,0501 Консервация,0,NaN,0.0,10.0,20,Куратор 3,Фирма 1,...,3.500,36.455696,0.214286,0.508318,0.171017,0,0,0,0,0
1,226249,Товар 95599,0501 Консервация,0,NaN,0.0,10.0,72,Куратор 3,Фирма 2,...,4.750,36.160000,0.157895,0.512474,0.126355,0,0,0,0,0
2,226249,Товар 95599,0501 Консервация,0,NaN,0.0,10.0,81,Куратор 3,Фирма 2,...,5.625,36.056440,0.133333,0.513946,0.106802,0,0,0,1,1
5,226249,Товар 95599,0501 Консервация,0,NaN,0.0,10.0,102,Куратор 3,Фирма 3,...,6.375,38.763966,0.117647,0.478049,0.091990,0,0,0,0,0
6,226249,Товар 95599,0501 Консервация,0,NaN,0.0,10.0,104,Куратор 3,Фирма 4,...,4.000,106.000000,0.187500,0.174821,0.104841,0,0,0,0,0


Теперь нужно посмотреть финальное распределение товаров

In [41]:
coef_items = pd.concat([alcohol_items, nonalcohol_items]).filter(items=[col_item, col_store, col_maxcap, col_remain, col_best])

dist_items = coef_items[coef_items[col_best] == 1].groupby([col_item]).first()  # на случай, если каким-то образом коэф. совпали

item_verdict = item_verdict.merge(dist_items, left_index=True, right_on=col_item, how="left").set_index(keys=col_item)

item_verdict.loc[item_verdict[col_best] == 1, col_verdict] = "Распределение найдено"

item_verdict = item_verdict.drop(columns=[col_best])

item_verdict[col_verdict] = item_verdict[col_verdict].fillna("Нет распределения")
dist_found = len(item_verdict[item_verdict[col_verdict] == "Распределение найдено"])
print("Найдено распределений:", dist_found)
print("Не распределено:", item_verdict.shape[0] - dist_found)
item_verdict.head()

Найдено распределений: 5044
Не распределено: 1357


,Номенклатура,Уровень3,Номенклатура.Квант поставки,Закрывающийся магазин,Объём товара в закрывающемся магазине (V),Итоговый вердикт,Магазин,Максимальная вместимость товара в магазине (M),Остаток товара в магазине (R)
Код товара,,,,,,,,,
226249,Товар 95599,0501 Консервация,0.0,10,10.0,Распределение найдено,130,8.0,6.0
234761,Товар 104090,0901 Сырье СП,0.0,10,3.0,Распределение найдено,50,7.0,7.0
215959,Товар 85472,"0101 Фрукты, Овощи, Зелень, Грибы",0.0,10,53.0,Товар имеет срок годности меньше 30 дней,NaN,NaN,NaN
237251,Товар 106571,"0904 Тара, Упаковка и прочее",0.0,10,3.0,Нет распределения,NaN,NaN,NaN
217430,Товар 86938,0705 Товары для уборки,3.0,10,12.0,Распределение найдено,130,8.0,8.0


In [42]:
closing_store_full = pd.concat([alcohol_items, nonalcohol_items])

Выгрузка таблиц в csv

In [43]:
output_csv(alcohol_items, "closing_store_alcohol_items_distribution.csv", False)
output_csv(nonalcohol_items, "closing_store_nonalcohol_items_distribution.csv", False)
output_csv(closing_store_full, "closing_store_full.csv", False)
output_csv(item_verdict, "total_item_distribution.csv", True)